# 🎯 Pokémon Kaggle Competition: Baseline Model
Welcome to the Pokémon Kaggle Competition! 
In this notebook, we will build a **Multi-Modal Neural Network** using Keras. 
Our goal is to predict if a Pokémon is **Legendary** (1 = Yes, 0 = No) by combining two types of data:
1. **Numerical Stats:** (HP, Attack, Defense, etc.)
2. **Image Data:** (The official sprite of the Pokémon)

Let's build a Neural Network that handles both!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Concatenate
from tensorflow.keras.preprocessing.image import img_to_array
from sklearn.preprocessing import StandardScaler

## 1. Load the Data
We will load `train.csv` and `test.csv`, and then write a function to load the corresponding images.

In [ ]:
# Load tabular data
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# Numerical features we want to use
num_features = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed', 'Weight', 'Height']

# Scale numerical features
scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[num_features])
X_test_num = scaler.transform(test_df[num_features])

y_train = train_df['Is_Legendary'].values

In [ ]:
# Function to load and resize images
def load_images(image_files, base_path='data/images/', target_size=(64, 64)):
    images = []
    for f in image_files:
        path = os.path.join(base_path, f)
        img = Image.open(path).convert('RGB') # Ensure 3 channels
        img = img.resize(target_size)
        img_array = img_to_array(img) / 255.0 # Normalize pixels to 0-1
        images.append(img_array)
    return np.array(images)

print("Loading Train Images...")
X_train_img = load_images(train_df['Image_File'])
print("Loading Test Images...")
X_test_img = load_images(test_df['Image_File'])

print(f"Train Images Shape: {X_train_img.shape}")

## 2. Build the Multi-Modal Neural Network
We will create a Keras Functional API model.
- **Branch 1:** Flattens the image and passes it through a Dense layer.
- **Branch 2:** Passes the numerical stats through a Dense layer.
- **Merge:** Concatenates both branches into a final set of Dense layers to predict 0 or 1.

In [ ]:
# -- Branch 1: Image Input --
img_input = Input(shape=(64, 64, 3), name='Image_Input')
x1 = Flatten()(img_input)
x1 = Dense(128, activation='relu')(x1)
x1 = Dense(64, activation='relu')(x1)

# -- Branch 2: Numerical Stats Input --
num_input = Input(shape=(len(num_features),), name='Stats_Input')
x2 = Dense(32, activation='relu')(num_input)

# -- Merge Both Branches --
merged = Concatenate()([x1, x2])
out = Dense(32, activation='relu')(merged)
output = Dense(1, activation='sigmoid', name='Prediction')(out) # Binary classification

# Compile the model
model = Model(inputs=[img_input, num_input], outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train the model
history = model.fit(
    x=[X_train_img, X_train_num], 
    y=y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

## 3. Generate Submission File
Now we predict on the test set and save the results into the Kaggle format.

In [ ]:
# Predict probabilities
test_preds = model.predict([X_test_img, X_test_num])

# Convert probabilities to 0 or 1 (threshold 0.5)
test_preds_binary = (test_preds > 0.5).astype(int).flatten()

# Prepare submission dataframe
submission = pd.read_csv('data/sample_submission.csv')
submission['Is_Legendary'] = test_preds_binary

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created! Ready to upload to Kaggle.")